# Agilent LCR & Janis Temperature Controller
Automated Measurements with Live Plotting

In [ ]:
# Imports
from nfoinstruments.drivers import Janis, E4890A
from nfoinstruments.drivers.setup import MeasurementSetup
from nfoinstruments.procedures import (
    set_temperature_and_wait, 
    sweep_frequency_lcr,
    single_frequency_time_scan,
    set_bias_and_wait,
    build_cv_bias_path,
    sweep_cv_lcr,
    load_measurement_files,
    load_cv_measurement_files,
    plot_all_measurements,
    plot_measurement_comparison,
    plot_time_scan_comparison,
    plot_cv_comparison
)
from nfoinstruments.procedures.utils import (
    run_temperature_bias_sweep_with_live_plot, 
    run_cv_sweep_with_live_plot,
    run_time_scan_with_live_plot,
    plot_is_overlay,
    plot_cv_overlay,
    plot_time_scan_overlay
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

# Output configuration
output_path = Path(r"C:\Users\F110216\Desktop\Data_Horatio\NewExperiment")

In [ ]:
# Initialize instruments
mm = MeasurementSetup()

# Connect to devices (update GPIB addresses if needed)
mm.connect_to_devices({
    'GPIB1::16::INSTR': Janis,
    'GPIB1::17::INSTR': E4890A
})

janis = mm.devices['GPIB1::16::INSTR']
lcr = mm.devices['GPIB1::17::INSTR']

# Configure LCR meter
lcr.measurement_type = E4890A.MeasurementType.ZTD  # Options: ZTD (Z, θ), RX (R, X), CPD (Cp, D), CSD (Cs, D)...
lcr.measurement_time = E4890A.MeasurementTime.LONG # Options: SHORT, MEDIUM, LONG
lcr.averages = 1                                 # 1 to 256
lcr.signal_type = E4890A.SignalType.VOLTAGE        # VOLTAGE or CURRENT
lcr.alc_enabled = False                            # Automatic Level Control (True/False) - MUST BE FALSE TO PROTECT MEMRISTOR

# Initialize into Standby Mode (Protect Sample during probe landing)
print("Initializing LCR into STANDBY MODE...")
lcr.signal_amplitude = 0.0                       # Signal Amplitude (V or A)
lcr.bias = 0.0                                     # DC Bias (V)
lcr.trigger_source = 'BUS'                        # Stop internal measurements (protects from bridge overload)

print(f"LCR configured:")
print(f"  Type: {lcr.measurement_type.name}")
print(f"  Time: {lcr.measurement_time.name} (Avg: {lcr.averages})")
print(f"  ALC: {lcr.alc_enabled}")
print(f"  AC Signal: {lcr.signal_amplitude} V")
print(f"  DC Bias: {lcr.bias} V")
print(f"  Trigger: {lcr.trigger_source}")

## Manual LCR & Temperature Control
Run this cell to manually activate/deactivate the LCR output or manually set the Janis temperature.

In [ ]:
# --- MANUAL STANDBY TOGGLE ---

# Uncomment to ACTIVATE signal
# lcr.signal_amplitude = 0.05
# lcr.trigger_source = 'INT'
# print(f"Signal ACTIVE: {lcr.signal_amplitude} V, Trigger: INT")

# Uncomment to DEACTIVATE signal (Standby)
# lcr.signal_amplitude = 0.0
# lcr.bias = 0.0
# lcr.trigger_source = 'BUS'
# print(f"Signal DEACTIVATED (Standby mode: 0V, BUS)")

# --- MANUAL TEMPERATURE CONTROL ---
# Uncomment to manually set Janis temperature
# set_temperature_and_wait(janis, target_temperature=300, extra_settle_time=30, verbose=True)


## 1. Initial (Pristine) IS Sweep
Do a single sweep at the current temperature before the main loop.

In [ ]:
# Define parameters for Pristine Sweep
# 60 points per decade, from 20 Hz to 2 MHz (5 decades -> 300 points)
frequencies = np.round(np.logspace(np.log10(20), np.log10(2e6), num=300), decimals=2)

current_temp = janis.temperature
dc_bias = 0.0  # V
ac_amplitude = 0.05 # V

sweep_name = "Long_Measurement_Exp_01"

# Run pristine measurement
next_run = run_temperature_bias_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name=sweep_name,
    temp_points=[current_temp],
    bias_points=[dc_bias],
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    freq_points=frequencies,
    Vrms=ac_amplitude,
    filename_suffix="_pristine", 
    run_count_start=1,
    run_select=None,
    extra_settle_time=0,
    log_y_left=True,   # log scale for Magnitude / Modulus Real
    log_y_right=False  # linear scale for Phase / Modulus Imaginary
)

### Plot IS Measurements (Standalone)
Run this to re-plot without taking new measurements.

In [ ]:
# You can load plot_data manually, but since run_temperature_bias_sweep_with_live_plot already
# plots existing runs matching 'run_select', you can just call the sweep function with an empty temp array!
# Example: Plot only run 1 from the folder.
plot_is_overlay_data = [] # This would need the data loader logic if you want to use the pure plotter.
# A quick trick is to call the measurement loop with empty parameters just to trigger the loader/plotter:
_ = run_temperature_bias_sweep_with_live_plot(
    parent_dir=output_path, sweep_name=sweep_name,
    temp_points=[], bias_points=[], janis_ctrl=janis, lcr_ctrl=lcr, freq_points=frequencies,
    run_count_start=next_run, run_select=None, log_y_left=True, log_y_right=False
)


## 2. Main Temperature & Bias Loop (IS Scans)

In [ ]:
# Define parameters for IS Sweep
temperatures = list(range(300, 140, -10))
dc_biases = [-2.0, -1.0, 0.0, 1.0, 2.0]  # V
ac_amplitude = 0.05 # V

# Run measurement
next_run = run_temperature_bias_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name=sweep_name,
    temp_points=temperatures,
    bias_points=dc_biases,
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    freq_points=frequencies,
    Vrms=ac_amplitude,
    filename_suffix="", 
    run_count_start=next_run,
    run_select=None,
    extra_settle_time=30,
    log_y_left=True,
    log_y_right=False
)

## 3. Capacitance-Voltage (CV) Scans

In [ ]:
# Define parameters for CV Sweep
Vmin = -5.0
Vmax = 5.0
Vstep = 0.5
Vrms = 0.1
cv_freq_points = [1e4, 1e5]
cycles = 1 

# Run measurement
next_run = run_cv_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name="CV_" + sweep_name,
    temp_points=temperatures,
    freq_points=cv_freq_points,
    Vmin=Vmin,
    Vmax=Vmax,
    Vstep=Vstep,
    Vrms=Vrms,
    cycles=cycles,
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    filename_suffix="",
    run_count_start=next_run,
    run_select=None,
    extra_settle_time=30,
    log_y_left=False,
    log_y_right=False
)

### Plot CV Measurements (Standalone)

In [ ]:
# Trigger CV loader/plotter by passing empty temps
_ = run_cv_sweep_with_live_plot(
    parent_dir=output_path, sweep_name="CV_" + sweep_name,
    temp_points=[], freq_points=cv_freq_points, Vmin=0, Vmax=0, Vstep=1, Vrms=0, cycles=0,
    janis_ctrl=janis, lcr_ctrl=lcr,
    run_count_start=next_run, run_select=None, log_y_left=False, log_y_right=False
)

## 4. Time Scan (Drift) Measurement
Hold a DC voltage and monitor impedance over time.

In [ ]:
# Define parameters for Time Scan
scan_duration = 120 # Seconds to monitor
update_interval = 2.0 # Seconds between live plot updates

ts_freqs = [1e4]      # Hz
ts_Vrms = [0.05]      # V
ts_Vdc = [0.0]        # V

# Run measurement
next_run = run_time_scan_with_live_plot(
    parent_dir=output_path,
    sweep_name="TimeScan_" + sweep_name,
    freq_points=ts_freqs,
    Vdc_points=ts_Vdc,
    Vrms_points=ts_Vrms,
    scan_duration=scan_duration,
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    update_interval=update_interval,
    filename_suffix="",
    run_count_start=next_run,
    run_select=None,
    log_y_left=False,  # Z Magnitude (Linear by default here)
    log_y_right=False  # Phase
)

### Plot Time Scan (Standalone)

In [ ]:
# Trigger Time Scan loader/plotter by passing empty points
_ = run_time_scan_with_live_plot(
    parent_dir=output_path, sweep_name="TimeScan_" + sweep_name,
    freq_points=[], Vdc_points=[], Vrms_points=[], scan_duration=0,
    janis_ctrl=janis, lcr_ctrl=lcr, run_count_start=next_run, run_select=None,
    log_y_left=False, log_y_right=False
)